# clrcycle on circadian mouse liver

[Open in Colab](https://colab.research.google.com/github/pachterlab/clrcycle/blob/main/tutorial/circadian_liver.ipynb)

This notebook shows how to fit `clrcycle` and create its standard two-panel figure using the GSE54650 mouse liver circadian dataset. No local data are required.

The workflow is divided into three clearly marked stages:

1. **Data processing:** download GEO data and create a samples-by-genes matrix.
2. **Feature selection:** select 240 rhythmic genes for this demonstration.
3. **clrcycle:** run `fit(...)` and `plot(...)`.

The input to `clrcycle` is a pandas data frame with **samples in rows, features in columns, and nonnegative values**. The notebook saves the resulting tables and figure under `tutorial/results/`.

## Setup

Run the next cell once per Colab session. It installs clrcycle only when the package is not already available.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("clrcycle") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/pachterlab/clrcycle.git",
    ])

## 1. Data processing

Everything in this section prepares the public liver data; these are not clrcycle commands. First, import the required packages and download the GEO expression matrix and probe annotation. Existing downloads are reused.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import gzip
import os
import re

import numpy as np
import pandas as pd
from clrcycle import fit, plot

tutorial_dir = Path(".") if Path.cwd().name == "tutorial" else Path("tutorial")
data_dir = Path(os.environ.get("CLRCYCLE_TUTORIAL_DATA", tutorial_dir / "data"))
data_dir.mkdir(parents=True, exist_ok=True)
matrix_path = data_dir / "GSE54650_series_matrix.txt.gz"
annotation_path = data_dir / "GPL6246.annot.gz"

if not matrix_path.exists():
    urlretrieve("https://ftp.ncbi.nlm.nih.gov/geo/series/GSE54nnn/GSE54650/matrix/GSE54650_series_matrix.txt.gz", matrix_path)
if not annotation_path.exists():
    urlretrieve("https://ftp.ncbi.nlm.nih.gov/geo/platforms/GPL6nnn/GPL6246/annot/GPL6246.annot.gz", annotation_path)

### Create the samples-by-genes matrix

Read the downloaded files, retain the 24 liver samples, map microarray probes to gene symbols, remove unusable values, and average probes that map to the same gene.

The output is `liver_genes`: a nonnegative data frame with one sample per row and one gene per column. `ct` contains the known collection times and is used only during feature selection.

In [ ]:
def table_bounds(path, begin_marker, end_marker):
    begin = end = None
    with gzip.open(path, "rt", encoding="utf-8", errors="replace") as handle:
        for line_number, line in enumerate(handle):
            if line.startswith(begin_marker):
                begin = line_number
            elif line.startswith(end_marker):
                end = line_number
                break
    return begin, end

def read_expression(path):
    titles = None
    with gzip.open(path, "rt", encoding="utf-8", errors="replace") as handle:
        for line in handle:
            if line.startswith("!Sample_title"):
                titles = [value.strip().strip('\"') for value in line.rstrip().split("\t")[1:]]
                break
    begin, end = table_bounds(path, "!series_matrix_table_begin", "!series_matrix_table_end")
    frame = pd.read_csv(path, sep="\t", compression="gzip", skiprows=begin + 1, nrows=end - begin - 2, dtype={"ID_REF": str})
    frame = frame.rename(columns={frame.columns[0]: "ID_REF"}).set_index("ID_REF")
    frame.columns = titles
    return frame.astype(float)

def read_annotation(path):
    begin, end = table_bounds(path, "!platform_table_begin", "!platform_table_end")
    frame = pd.read_csv(path, sep="\t", compression="gzip", skiprows=begin + 1, nrows=end - begin - 2, dtype=str, low_memory=False)
    frame = frame.rename(columns={"ID": "ID_REF", "Gene symbol": "gene_symbol"})
    return frame[["ID_REF", "gene_symbol"]].dropna().drop_duplicates("ID_REF")

expression = read_expression(matrix_path)
annotation = read_annotation(annotation_path)
liver_samples = sorted(
    [name for name in expression.columns if name.startswith("Liv_CT")],
    key=lambda name: int(re.search(r"CT(\d+)", name).group(1)),
)
liver = expression[liver_samples].T
ct = np.array([int(re.search(r"CT(\d+)", name).group(1)) for name in liver.index])

positive = np.isfinite(liver).all(axis=0) & (liver.min(axis=0) > 0)
liver = liver.loc[:, positive]
probe_map = annotation.set_index("ID_REF").reindex(liver.columns)["gene_symbol"]
mapped = probe_map.notna() & probe_map.str.len().gt(0)
liver_genes = liver.loc[:, mapped].T.assign(gene_symbol=probe_map[mapped]).groupby("gene_symbol").mean().T
liver_genes.shape

## 2. Feature selection

This section is specific to the circadian liver example; it is not a clrcycle command. We score each gene by how well its log expression follows a 24-hour sine/cosine curve and retain the top 240 genes. This produces `panel`, the samples-by-features matrix passed to clrcycle.

For your own dataset, choose features in a way appropriate for your experiment. You may also pass a wide matrix directly to `fit`, which selects high-log-variance features without using sample labels.

In [ ]:
phase = 2 * np.pi * (ct % 24) / 24
design = np.column_stack([np.cos(phase), np.sin(phase)])
log_expression = np.log(liver_genes.to_numpy())
centered = log_expression - log_expression.mean(axis=0, keepdims=True)
fitted = design @ np.linalg.lstsq(design, centered, rcond=None)[0]
harmonic_r2 = np.divide(
    np.sum(fitted**2, axis=0),
    np.sum(centered**2, axis=0),
    out=np.zeros(liver_genes.shape[1]),
    where=np.sum(centered**2, axis=0) > 0,
)
panel = liver_genes.loc[:, liver_genes.columns[np.argsort(harmonic_r2)[-240:]]]
panel.shape

## 3. Run clrcycle

These are the actual clrcycle commands. `fit` learns the circular gene order and projects the samples. `plot` turns that fit result into a single figure containing the sample projection and learned gene circle.

Because feature selection is already complete, `max_features=None` tells clrcycle to use all 240 columns in `panel`. The optional `feature_labels` list controls which gene names are written around the circle; it does not affect the fit.

In [ ]:
result = fit(panel, max_features=None)

clock_genes = ["Arntl", "Clock", "Cry1", "Cry2", "Per1", "Per2", "Per3", "Nr1d1", "Dbp", "Rorc", "Nampt"]
figure = plot(result, feature_labels=clock_genes)
figure

## Save the results

A clrcycle fit contains three ordinary pandas tables. `result.coordinates` describes the samples, `result.feature_order` describes the learned feature circle, and `result.feature_weights` gives each feature's contribution to each sample's radius. The following cell saves the tables and figure.

In [ ]:
results_dir = tutorial_dir / "results"
results_dir.mkdir(parents=True, exist_ok=True)
result.coordinates.to_csv(results_dir / "liver_sample_coordinates.csv", index=False)
result.feature_order.to_csv(results_dir / "liver_feature_order.csv", index=False)
result.feature_weights.to_csv(results_dir / "liver_feature_weights.csv", index=False)
figure.savefig(results_dir / "liver_clrcycle.png", dpi=220)
figure.savefig(results_dir / "liver_clrcycle.svg")
print(f"Saved tutorial outputs to {results_dir.resolve()}")

## Interpret the figure

In the **sample projection** (left), each point is one liver sample. Its angle gives its position around the learned cycle, and its radius (distance from the origin) measures the strength of its circular signal. For ordered centered CLR value $Z_{ig}$ and the fitted normalized cosine and sine basis values $c_g$ and $s_g$, the per-feature weight is $w_{ig}=Z_{ig}(c_g\cos(\phi_i)+s_g\sin(\phi_i))$. `feature_weight` is feature $g$'s contribution to sample $i$'s radial first-Fourier projection: positive values reinforce its projected direction, negative values oppose it, and they sum exactly to the sample radius. These weights come from centered CLR values and fitted first-Fourier geometry; they are not expression values or differential-expression statistics.

In the **learned feature circle** (right), nearby genes occupy similar positions in the learned cyclic order. Color represents clrcycle angle in both panels. Labels are shown only for the requested clock genes that occur in the fitted panel.

The absolute orientation of an unsupervised circle is arbitrary: rotating or reflecting the entire result does not change the learned cyclic structure. Align it to known time or another reference only when an absolute phase interpretation is needed.

## Use clrcycle with your own data

Once you have a nonnegative samples-by-features table, the reusable workflow is:

```python
data = pd.read_csv("my_expression_matrix.csv", index_col=0)
result = fit(data)
result.coordinates
result.feature_order
result.feature_weights
figure = plot(result)
```

Use the result tables for downstream analysis or metadata-aware figures. For example, `result.feature_weights.query("sample == 'sample_id'").nlargest(10, "feature_weight")` retrieves one sample's largest reinforcing contributions. Publication-specific examples are available in the [clrcycle paper repository](https://github.com/pachterlab/SEP_2026).